# Preventra &mdash; Phase 1: Discharge-Time Readmission Risk (MIMIC-IV)

Trains a **30-day unplanned readmission** model on the MIMIC-IV `hosp` module.

### Why this notebook is written the way it is

`labevents.csv` is **18.4 GB** and `prescriptions.csv` is **3.5 GB**. A Kaggle CPU session has
**32 GB RAM**, and pandas needs roughly 3&ndash;5&times; a CSV's disk size once parsed. Loading
either table directly will kill the kernel.

Every table above ~500 MB is therefore **streamed in chunks**, filtered to the study cohort,
reduced to per-admission aggregates, and discarded. Nothing large is ever held whole.

### Pipeline
| Step | Source | Size | Strategy |
|---|---|---|---|
| Cohort + label | `admissions`, `patients` | 106 MB | direct load |
| Comorbidity (CCI) | `diagnoses_icd` | 181 MB | direct load, **ICD-9 + ICD-10** |
| Severity | `drgcodes` | 55 MB | direct load (APR-DRG) |
| Procedures | `procedures_icd` | 34 MB | direct load |
| Medications | `prescriptions` | **3.5 GB** | chunked scan |
| Labs | `labevents` | **18.4 GB** | chunked scan + partial aggregation |

### Two correctness guards worth calling out
1. **Split by patient, not by admission.** Patients have multiple stays; a random row split
   leaks the same person into train and test and inflates every metric.
2. **Calibrate the output.** Readmission is ~10% prevalence. An uncalibrated model trained on
   rebalanced data emits probabilities several times higher than reality, which breaks any
   downstream cost/ROI calculation that treats the score as a real probability.

## 0 &middot; Setup and memory helpers

In [ ]:
import os, gc, re, glob, json, warnings, time
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

# --- Kaggle resource envelope -------------------------------------------------
# CPU session: ~32 GB RAM | /kaggle/working ~20 GB (persistent)
#              /kaggle/temp ~60 GB (scratch)  | /kaggle/input read-only
#
# labevents.csv is 18.4 GB on disk. A naive read_csv would need roughly 3-5x
# that in RAM once parsed, so it is NEVER loaded whole anywhere in this
# notebook. Every table over ~500 MB is streamed in chunks, filtered down to
# the study cohort, aggregated, and discarded.
# -----------------------------------------------------------------------------

CHUNK_ROWS     = 2_000_000   # rows per chunk; lower this if you still hit memory pressure
COMPACT_EVERY  = 8           # re-aggregate partial results every N chunks
CACHE_DIR      = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./cache"
os.makedirs(CACHE_DIR, exist_ok=True)

def mem_mb(df):
    return df.memory_usage(deep=True).sum() / 1024**2

def report(label, df):
    print(f"  {label:<34} rows={len(df):>12,}  cols={df.shape[1]:>3}  mem={mem_mb(df):>8.1f} MB")

def downcast(df):
    """Shrink numeric columns in place. Typically halves memory."""
    for c in df.select_dtypes(include=["int64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")
    for c in df.select_dtypes(include=["float64"]).columns:
        df[c] = pd.to_numeric(df[c], downcast="float")
    return df

def cached(name):
    """Decorator-free cache helper: returns path for a parquet cache file."""
    return os.path.join(CACHE_DIR, f"{name}.parquet")

def load_cache(name):
    p = cached(name)
    if os.path.exists(p):
        print(f"  [cache hit] {name}")
        return pd.read_parquet(p)
    return None

def save_cache(df, name):
    df.to_parquet(cached(name), index=False)
    print(f"  [cached] {name}  ({os.path.getsize(cached(name))/1024**2:.1f} MB)")

In [ ]:
def find_mimic_root():
    """Locate the MIMIC-IV hosp directory on Kaggle or locally."""
    bases = ["/kaggle/input", "./data", ".", ".."]
    hits = []
    for base in bases:
        if not os.path.isdir(base):
            continue
        for depth in range(4):
            for stem in ("admissions.csv*", "sample_admissions.csv*"):
                pattern = os.path.join(base, *(["*"] * depth), stem)
                hits += [os.path.dirname(p) for p in glob.glob(pattern)]
    # Prefer a directory that also carries the other tables we need
    for h in sorted(set(hits), key=len):
        if glob.glob(os.path.join(h, "diagnoses_icd.csv*")) or \
           glob.glob(os.path.join(h, "sample_diagnoses_icd.csv*")):
            return h
    return hits[0] if hits else None

MIMIC = find_mimic_root()
if MIMIC is None:
    raise FileNotFoundError(
        "Could not find admissions.csv. On Kaggle, attach the MIMIC-IV dataset "
        "and it will mount under /kaggle/input/<slug>/."
    )
print("MIMIC-IV hosp directory:", MIMIC)

def tbl(name):
    """Resolve a table name to an actual path (.csv, .csv.gz, or demo sample_ prefix)."""
    for cand in (f"{name}.csv", f"{name}.csv.gz", f"sample_{name}.csv"):
        p = os.path.join(MIMIC, cand)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} not found in {MIMIC}")

def size_gb(name):
    try:
        return os.path.getsize(tbl(name)) / 1024**3
    except FileNotFoundError:
        return 0.0

print("\ntable sizes on disk:")
for t in ["admissions","patients","diagnoses_icd","drgcodes","procedures_icd",
          "omr","prescriptions","labevents","d_labitems"]:
    g = size_gb(t)
    tag = "  <-- STREAMED" if g > 0.5 else ""
    print(f"  {t:<18} {g:>8.2f} GB{tag}")

## 1 &middot; Cohort and label

In [ ]:
# ---------------------------------------------------------------------------
# Cohort + 30-day unplanned readmission label
# ---------------------------------------------------------------------------
# Exclusions follow standard readmission-measure practice:
#   * died in hospital / discharged to hospice -> cannot be readmitted
#   * observation stays  -> not true inpatient admissions (CMS excludes them)
#   * elective returns   -> planned follow-up, not a care failure
#
# The "next admission" is computed on the FULL sequence BEFORE any rows are
# dropped. Filtering first would lose a readmission that happened to follow an
# excluded stay.
# ---------------------------------------------------------------------------

TERMINAL_DISPOSITIONS = {"DIED", "HOSPICE"}
OBSERVATION_TYPES = {"EU OBSERVATION", "OBSERVATION ADMIT",
                     "AMBULATORY OBSERVATION", "DIRECT OBSERVATION"}
READMIT_WINDOW_DAYS = 30

adm = pd.read_csv(
    tbl("admissions"),
    usecols=["subject_id","hadm_id","admittime","dischtime","admission_type",
             "admission_location","discharge_location","insurance","marital_status",
             "race","edregtime","edouttime","hospital_expire_flag"],
    parse_dates=["admittime","dischtime","edregtime","edouttime"],
)
# anchor_year_group is the TRUE 3-year range that this patient's shifted
# anchor_year corresponds to. It is the only key back to real calendar time,
# and section 6b uses it to undo the de-identification date shift.
patients = pd.read_csv(tbl("patients"),
                  usecols=["subject_id","gender","anchor_age","anchor_year",
                           "anchor_year_group","dod"],
                  parse_dates=["dod"])
report("admissions", adm); report("patients", patients)

adm = adm.sort_values(["subject_id","admittime"]).reset_index(drop=True)

# --- prior utilisation, computed from the sequence itself --------------------
adm["prev_dischtime"] = adm.groupby("subject_id")["dischtime"].shift(1)
adm["n_prior_adm"] = adm.groupby("subject_id").cumcount()
adm["days_since_prev"] = (adm.admittime - adm.prev_dischtime).dt.total_seconds()/86400

# --- forward-looking label ---------------------------------------------------
adm["next_admittime"] = adm.groupby("subject_id")["admittime"].shift(-1)
adm["next_type"]      = adm.groupby("subject_id")["admission_type"].shift(-1)
adm["days_to_next"]   = (adm.next_admittime - adm.dischtime).dt.total_seconds()/86400

died        = adm.hospital_expire_flag.eq(1)
terminal    = adm.discharge_location.isin(TERMINAL_DISPOSITIONS)
observation = adm.admission_type.isin(OBSERVATION_TYPES)

print(f"\nexclusions from {len(adm):,} admissions")
print(f"  died in hospital      : {died.sum():>8,}")
print(f"  hospice/died at disch : {terminal.sum():>8,}")
print(f"  observation stays     : {observation.sum():>8,}")

coh = adm[~(died | terminal | observation)].copy()

within    = coh.days_to_next.between(0, READMIT_WINDOW_DAYS)
unplanned = ~coh.next_type.eq("ELECTIVE")
coh["readmit_30d"] = (within & unplanned).astype("int8")

print(f"  -> eligible index stays: {len(coh):>8,}  ({len(coh)/len(adm):.1%})")
print(f"\n{READMIT_WINDOW_DAYS}-day unplanned readmission rate: "
      f"{coh.readmit_30d.sum():,} / {len(coh):,} = {coh.readmit_30d.mean():.2%}")
print(f"distinct patients: {coh.subject_id.nunique():,}")

BASE_RATE = float(coh.readmit_30d.mean())
del adm; gc.collect()

## 2 &middot; Comorbidity &mdash; Charlson index across **both** ICD versions

MIMIC-IV diagnoses are split roughly 50/50 between ICD-9 and ICD-10. A mapping written for
only one version silently scores `0` for every row in the other, understating comorbidity for
half the cohort with no error raised.

Both mappings below follow **Quan et al. (2005)**, the standard adaptation of Charlson for
administrative data.

In [ ]:
# Quan et al. (2005) Charlson mappings, as regex prefixes.
# weight -> (icd9_pattern, icd10_pattern)
CHARLSON = {
    "myocardial_infarction":   (1, r"^(410|412)",                         r"^(I21|I22|I252)"),
    "congestive_heart_failure":(1, r"^(39891|402(01|11|91)|404(01|03|11|13|91|93)|425[456789]|428)",
                                   r"^(I099|I110|I130|I132|I255|I420|I42[56789]|I43|I50|P290)"),
    "peripheral_vascular":     (1, r"^(0930|437[34]|440|441|443[123456789]|4471|5571|5579|V434)",
                                   r"^(I70|I71|I73[189]|I771|I79[02]|K55[189]|Z95[89])"),
    "cerebrovascular":         (1, r"^(36234|43[0-8])",                   r"^(G4[56]|H340|I6[0-9])"),
    "dementia":                (1, r"^(290|2941|3312)",                   r"^(F0[0-3]|F051|G30|G311)"),
    "chronic_pulmonary":       (1, r"^(4168|4169|49[0-6]|50[0-5]|5064|5081|5088)",
                                   r"^(I27[89]|J4[0-7]|J6[0-7]|J684|J70[13])"),
    "rheumatic":               (1, r"^(4465|710[0-4]|714[0128]|725)",     r"^(M0[56]|M315|M3[234]|M35[13]|M360)"),
    "peptic_ulcer":            (1, r"^(53[1-4])",                         r"^(K2[5-8])"),
    "mild_liver":              (1, r"^(070[23][23]|070[45]4|0706|0709|57[01]|5733|5734|5738|5739|V427)",
                                   r"^(B18|K70[0-39]|K71[3-57]|K7[34]|K76[02-49]|Z944)"),
    "diabetes_uncomplicated":  (1, r"^(250[0-3]|2508|2509)",              r"^(E1[0-4][01689])"),
    "diabetes_complicated":    (2, r"^(250[4-7])",                        r"^(E1[0-4][2-57])"),
    "hemiplegia":              (2, r"^(3341|342|343|344[0-6]|3449)",      r"^(G041|G114|G80[12]|G8[12]|G83[0-49])"),
    "renal_disease":           (2, r"^(40301|40311|40391|40402|40403|40412|40413|40492|40493|58[26]|5830[0-7]|585|586|5880|V420|V451|V56)",
                                   r"^(I120|I131|N03[2-7]|N05[2-7]|N1[89]|N250|Z49[0-2]|Z940|Z992)"),
    "malignancy":              (2, r"^(1[4-9][0-9]|20[0-8]|2386)",        r"^(C[0-2][0-6]|C3[0-4]|C3[789]|C4[01]|C43|C4[5-9]|C5[0-8]|C6[0-9]|C7[0-6]|C8[1-5]|C88|C9[0-7])"),
    "severe_liver":            (3, r"^(456[012]|572[2-8])",               r"^(I85[09]|I864|I982|K704|K711|K721|K729|K76[5-7])"),
    "metastatic_cancer":       (6, r"^(19[6-9])",                          r"^(C7[789]|C80)"),
    "hiv_aids":                (6, r"^(04[2-4])",                          r"^(B2[0-4])"),
}

dx = pd.read_csv(tbl("diagnoses_icd"), usecols=["hadm_id","icd_code","icd_version"])
dx = dx[dx.hadm_id.isin(set(coh.hadm_id))]
dx["icd_code"] = dx.icd_code.astype(str).str.upper().str.replace(".", "", regex=False).str.strip()
report("diagnoses (cohort only)", dx)
print("ICD version split:", dx.icd_version.value_counts(normalize=True).round(3).to_dict())

is9 = dx.icd_version.eq(9)
cci = pd.DataFrame({"hadm_id": coh.hadm_id.unique()}).set_index("hadm_id")

for cond, (w, p9, p10) in CHARLSON.items():
    hit = np.where(is9, dx.icd_code.str.match(p9), dx.icd_code.str.match(p10))
    hadms = dx.loc[hit, "hadm_id"].unique()
    cci[cond] = 0
    cci.loc[cci.index.isin(hadms), cond] = 1

# Charlson hierarchy: the severe form supersedes the mild one
cci.loc[cci.diabetes_complicated.eq(1), "diabetes_uncomplicated"] = 0
cci.loc[cci.severe_liver.eq(1), "mild_liver"] = 0
cci.loc[cci.metastatic_cancer.eq(1), "malignancy"] = 0

cci["charlson_score"] = sum(cci[c] * w for c, (w, _, _) in CHARLSON.items())
cci["n_diagnoses"] = dx.groupby("hadm_id").size().reindex(cci.index).fillna(0).astype("int16")
cci = downcast(cci.reset_index())

print(f"\ncharlson_score  mean={cci.charlson_score.mean():.2f}  median={cci.charlson_score.median():.0f}"
      f"  max={cci.charlson_score.max()}")
print(f"admissions with score 0: {cci.charlson_score.eq(0).mean():.1%}  "
      f"(a sanity check - if this is near 100% the mapping is broken)")
del dx; gc.collect()

## 3 &middot; Severity (APR-DRG) and procedure burden

In [ ]:
# APR-DRG carries an explicit 1-4 severity and mortality score: a far better
# acuity proxy than inferring severity from diagnosis codes alone.
drg = pd.read_csv(tbl("drgcodes"), usecols=["hadm_id","drg_type","drg_severity","drg_mortality"])
drg = drg[drg.hadm_id.isin(set(coh.hadm_id)) & drg.drg_type.eq("APR")]
drg = drg.groupby("hadm_id")[["drg_severity","drg_mortality"]].max().reset_index()
print(f"APR-DRG severity coverage: {drg.hadm_id.nunique()/coh.hadm_id.nunique():.1%} of cohort")

proc = pd.read_csv(tbl("procedures_icd"), usecols=["hadm_id","icd_code"])
proc = (proc[proc.hadm_id.isin(set(coh.hadm_id))]
        .groupby("hadm_id").size().rename("n_procedures").reset_index())
print(f"admissions with >=1 procedure: {len(proc)/coh.hadm_id.nunique():.1%}")
gc.collect()

## 4 &middot; Medications &mdash; streamed from `prescriptions.csv` (3.5 GB)

First streamed table. The pattern used here and for labs:

1. read a chunk with `usecols` only,
2. immediately drop rows outside the cohort,
3. reduce to per-admission aggregates,
4. append the (small) aggregate and free the chunk.

In [ ]:
COHORT_HADM = set(coh.hadm_id)

HIGH_RISK_PATTERNS = {
    "insulin":       r"insulin",
    "anticoagulant": r"warfarin|heparin|apixaban|rivaroxaban|enoxaparin|dabigatran",
    "opioid":        r"morphine|oxycodone|hydromorphone|fentanyl|hydrocodone|tramadol",
    "diuretic":      r"furosemide|bumetanide|torsemide|spironolactone|hydrochlorothiazide",
    "antipsychotic": r"haloperidol|quetiapine|olanzapine|risperidone",
}

cache = load_cache("phase1_meds")
if cache is not None:
    meds = cache
else:
    parts, t0 = [], time.time()
    reader = pd.read_csv(tbl("prescriptions"), usecols=["hadm_id","drug"],
                         chunksize=CHUNK_ROWS, dtype={"drug": "string"})
    for i, ch in enumerate(reader, 1):
        ch = ch[ch.hadm_id.isin(COHORT_HADM)]
        if ch.empty:
            continue
        ch["drug_l"] = ch.drug.str.lower()
        agg = ch.groupby("hadm_id").agg(n_drug_orders=("drug","size"),
                                        n_distinct_drugs=("drug","nunique"))
        for name, pattern in HIGH_RISK_PATTERNS.items():
            agg[f"med_{name}"] = ch.assign(f=ch.drug_l.str.contains(pattern, na=False, regex=True)) \
                                   .groupby("hadm_id").f.max().astype("int8")
        parts.append(agg.reset_index())
        if i % COMPACT_EVERY == 0:
            parts = [pd.concat(parts, ignore_index=True)
                       .groupby("hadm_id", as_index=False).max()]
            print(f"    chunk {i:>3} | {time.time()-t0:6.1f}s | admissions so far: {len(parts[0]):,}")
        del ch; gc.collect()

    if parts:
        meds = (pd.concat(parts, ignore_index=True)
                  .groupby("hadm_id", as_index=False)
                  .agg({"n_drug_orders":"sum", "n_distinct_drugs":"max",
                        **{f"med_{k}":"max" for k in HIGH_RISK_PATTERNS}}))
    else:
        # No prescription rows matched the cohort (possible on a truncated demo
        # extract). Emit the empty schema so the downstream merge still works.
        print("  WARNING: no prescription rows matched the cohort")
        meds = pd.DataFrame({"hadm_id": pd.Series(dtype="int64"),
                             "n_drug_orders": pd.Series(dtype="float64"),
                             "n_distinct_drugs": pd.Series(dtype="float64"),
                             **{f"med_{k}": pd.Series(dtype="float64")
                                for k in HIGH_RISK_PATTERNS}})
    meds = downcast(meds)
    save_cache(meds, "phase1_meds")
    del parts; gc.collect()

report("medication aggregates", meds)
print(meds[[c for c in meds.columns if c.startswith("med_")]].mean().round(3).to_string())

## 5 &middot; Labs &mdash; streamed from `labevents.csv` (18.4 GB)

The largest table by an order of magnitude. Two extra safeguards here:

- **Item resolution first.** `d_labitems` is 64 KB, so the target `itemid`s are resolved up
  front and the 18.4 GB scan filters on a tiny integer set rather than matching strings.
- **Periodic compaction.** Partial aggregates are re-reduced every few chunks so the
  accumulator cannot grow unbounded across the full pass.

Expect roughly 15&ndash;35 minutes for the full file. The result is cached to
`/kaggle/working`, so a re-run is instant.

In [ ]:
LAB_PATTERNS = {
    "hba1c":      r"hemoglobin a1c|hba1c|% *hemoglobin a1c",
    "glucose":    r"^glucose$",
    "creatinine": r"^creatinine$",
    "bun":        r"urea nitrogen",
    "sodium":     r"^sodium$",
    "potassium":  r"^potassium$",
    "hemoglobin": r"^hemoglobin$",
    "wbc":        r"white blood cells",
    "platelets":  r"platelet count",
    "albumin":    r"^albumin$",
    "bicarbonate":r"^bicarbonate$",
    "inr":        r"^inr\(pt\)$|international normalized",
}

dlab = pd.read_csv(tbl("d_labitems"))
dlab["label_l"] = dlab.label.astype(str).str.lower()

item_map = {}
for name, pattern in LAB_PATTERNS.items():
    m = dlab[dlab.label_l.str.contains(pattern, regex=True, na=False)]
    if "fluid" in m.columns:
        blood = m[m.fluid.eq("Blood")]
        m = blood if len(blood) else m
    for iid in m.itemid.tolist():
        item_map[int(iid)] = name
    print(f"  {name:<12} -> {len(m):>2} itemid(s)  {m.label.head(2).tolist()}")

TARGET_ITEMS = set(item_map)
print(f"\nresolved {len(TARGET_ITEMS)} itemids across {len(LAB_PATTERNS)} lab concepts")
if not TARGET_ITEMS:
    raise RuntimeError("No lab itemids resolved - check d_labitems labels")

In [ ]:
cache = load_cache("phase1_labs")
if cache is not None:
    labs = cache
else:
    parts, t0, seen = [], time.time(), 0
    reader = pd.read_csv(
        tbl("labevents"),
        usecols=["hadm_id","itemid","charttime","valuenum","flag"],
        parse_dates=["charttime"],
        chunksize=CHUNK_ROWS,
    )
    for i, ch in enumerate(reader, 1):
        seen += len(ch)
        ch = ch[ch.itemid.isin(TARGET_ITEMS) & ch.hadm_id.isin(COHORT_HADM) & ch.valuenum.notna()]
        if not ch.empty:
            ch["lab"] = ch.itemid.map(item_map)
            ch["abn"] = ch.flag.eq("abnormal").astype("int8")
            ch = ch.sort_values("charttime")
            agg = ch.groupby(["hadm_id","lab"]).agg(
                last_t=("charttime","max"),
                last_v=("valuenum","last"),
                min_v =("valuenum","min"),
                max_v =("valuenum","max"),
                n     =("valuenum","size"),
                n_abn =("abn","sum"),
            ).reset_index()
            parts.append(agg)
        if i % COMPACT_EVERY == 0 and parts:
            cat = pd.concat(parts, ignore_index=True).sort_values("last_t")
            parts = [cat.groupby(["hadm_id","lab"], as_index=False).agg(
                last_t=("last_t","max"), last_v=("last_v","last"),
                min_v=("min_v","min"), max_v=("max_v","max"),
                n=("n","sum"), n_abn=("n_abn","sum"))]
            print(f"    chunk {i:>3} | {seen/1e6:6.1f}M rows | {time.time()-t0:6.1f}s "
                  f"| kept {len(parts[0]):,}")
        del ch; gc.collect()

    if parts:
        long = (pd.concat(parts, ignore_index=True).sort_values("last_t")
                  .groupby(["hadm_id","lab"], as_index=False)
                  .agg(last_v=("last_v","last"), min_v=("min_v","min"),
                       max_v=("max_v","max"), n=("n","sum"), n_abn=("n_abn","sum")))
        # long -> wide, one column per (lab, statistic)
        labs = long.pivot(index="hadm_id", columns="lab",
                          values=["last_v","min_v","max_v","n_abn"])
        labs.columns = [f"lab_{lab}_{stat.replace('_v','')}" for stat, lab in labs.columns]
        labs = downcast(labs.reset_index())
        del long
    else:
        # No lab rows matched. Most likely causes: a truncated demo extract, or
        # itemid resolution failing against d_labitems. The model can still be
        # trained without labs, so emit an empty frame rather than aborting.
        print("  WARNING: no lab rows matched the cohort - continuing without lab features")
        labs = pd.DataFrame({"hadm_id": pd.Series(dtype="int64")})
    save_cache(labs, "phase1_labs")
    del parts; gc.collect()

report("lab aggregates", labs)
if labs.shape[1] > 1:
    cov = labs.notna().mean().sort_values(ascending=False)
    print("\ncoverage of the most-populated lab features:")
    print((cov.head(12) * 100).round(1).to_string())
else:
    print("\nno lab features produced (see warning above)")

## 6 &middot; Assemble the modelling matrix

In [ ]:
# Normalise the join key across every side table. downcast() may have made some
# int32 and an empty guard-frame would otherwise arrive as object, which makes
# the merge raise rather than simply producing NaNs.
for _df in (cci, drg, proc, meds, labs):
    if "hadm_id" in _df.columns:
        _df["hadm_id"] = _df["hadm_id"].astype("int64")

X = (coh[["subject_id","hadm_id","admittime","dischtime","admission_type",
          "admission_location","discharge_location","insurance","marital_status","race",
          "edregtime","edouttime","n_prior_adm","days_since_prev","readmit_30d"]]
     .merge(patients[["subject_id","gender","anchor_age"]], on="subject_id", how="left")
     .merge(cci,  on="hadm_id", how="left")
     .merge(drg,  on="hadm_id", how="left")
     .merge(proc, on="hadm_id", how="left")
     .merge(meds, on="hadm_id", how="left")
     .merge(labs, on="hadm_id", how="left"))

# --- derived features --------------------------------------------------------
X["los_days"]      = (X.dischtime - X.admittime).dt.total_seconds()/86400
X["ed_visit"]      = X.edregtime.notna().astype("int8")
X["ed_hours"]      = (X.edouttime - X.edregtime).dt.total_seconds()/3600
X["is_emergency"]  = X.admission_type.str.contains("EMER|URGENT", na=False).astype("int8")
X["n_procedures"]  = X.n_procedures.fillna(0)
X["n_drug_orders"] = X.n_drug_orders.fillna(0)
X["n_distinct_drugs"] = X.n_distinct_drugs.fillna(0)
X["prior_adm_flag"]= (X.n_prior_adm > 0).astype("int8")
X["readmit_history"] = (X.days_since_prev.notna() & (X.days_since_prev <= 30)).astype("int8")
for c in [c for c in X.columns if c.startswith("med_")]:
    X[c] = X[c].fillna(0)

# Discharge destination is a strong, legitimate discharge-time signal
X["disch_home"]  = X.discharge_location.eq("HOME").astype("int8")
X["disch_snf"]   = X.discharge_location.isin(["SKILLED NURSING FACILITY","REHAB",
                                              "CHRONIC/LONG TERM ACUTE CARE"]).astype("int8")
X["disch_ama"]   = X.discharge_location.eq("AGAINST ADVICE").astype("int8")

CATEGORICALS = ["gender","insurance","admission_type","marital_status","race"]
for c in CATEGORICALS:
    X[c] = X[c].fillna("UNKNOWN").astype("category")

DROP = ["hadm_id","admittime","dischtime","edregtime","edouttime",
        "discharge_location","admission_location"]
FEATURES = [c for c in X.columns if c not in DROP + ["subject_id","readmit_30d"]]

X = downcast(X)
report("modelling matrix", X)
print(f"\nfeatures: {len(FEATURES)}   positives: {X.readmit_30d.sum():,} "
      f"({X.readmit_30d.mean():.2%})")
# hadm_id is carried in the matrix as an IDENTIFIER, not a feature: it stays
# out of FEATURES so the model sees exactly what it saw before, but without
# it a scored row cannot be joined back to diagnoses, dates, or any other
# per-admission table downstream.
save_cache(X[["subject_id","hadm_id","readmit_30d"] + FEATURES], "phase1_matrix")

## 6b &middot; Dashboard exports

Three artefacts the Preventra dashboard needs that the modelling matrix cannot carry.
**None of them touch `FEATURES`, so the trained model is bit-for-bit unchanged.**

| output | why it is needed |
|---|---|
| `phase1_diagnoses.parquet` | what the patient was actually diagnosed with, so a risk score sits beside a clinical reason instead of a bare number |
| `phase1_timeline.parquet` | real admit/discharge dates, recovered from the de-identification shift rather than invented |

The diagnosis join uses the **`(icd_code, icd_version)` pair**. ICD-9 and ICD-10 code spaces
overlap &mdash; the same string is valid in both and means different things &mdash; so joining on
`icd_code` alone silently mislabels roughly half the cohort.


In [ ]:
# ---------------------------------------------------------------------------
# Export 1 - admission diagnoses
# ---------------------------------------------------------------------------
# seq_num == 1 is the PRINCIPAL diagnosis: the condition chiefly responsible for
# the admission. That is the headline the dashboard shows; the next few are kept
# as secondary context.
#
# Exported with titles already resolved so the dashboard never needs the 6M-row
# diagnoses_icd table at serving time.
# ---------------------------------------------------------------------------

N_SECONDARY = 3

dxf = pd.read_csv(tbl("diagnoses_icd"),
                  usecols=["hadm_id", "seq_num", "icd_code", "icd_version"])
dxf = dxf[dxf.hadm_id.isin(set(coh.hadm_id))]
dicd = pd.read_csv(tbl("d_icd_diagnoses"),
                   usecols=["icd_code", "icd_version", "long_title"])

# Normalise identically on both sides. MIMIC stores codes without dots, but
# releases differ on padding and case, and a one-sided normalisation produces a
# silent 100% join miss rather than an error.
for _f in (dxf, dicd):
    _f["icd_code"] = (_f.icd_code.astype(str).str.upper()
                      .str.replace(".", "", regex=False).str.strip())
    _f["icd_version"] = pd.to_numeric(_f.icd_version, errors="coerce").astype("Int64")

dicd = dicd.drop_duplicates(subset=["icd_code", "icd_version"])
dxf = dxf.merge(dicd, on=["icd_code", "icd_version"], how="left")

miss = dxf.long_title.isna().mean()
print(f"  codes with no dictionary entry: {miss:.2%}"
      f"{'   <-- CHECK THE JOIN' if miss > 0.05 else ''}")

# Keep the raw code rather than dropping the row: a dropped diagnosis would
# understate how many problems the admission actually carried.
dxf["long_title"] = dxf.long_title.fillna("Unmapped ICD code " + dxf.icd_code)
dxf = dxf.sort_values(["hadm_id", "seq_num"])

principal = dxf[dxf.seq_num == 1].drop_duplicates("hadm_id").set_index("hadm_id")
secondary = (dxf[dxf.seq_num > 1].groupby("hadm_id").long_title
             .apply(lambda s: list(s.head(N_SECONDARY))))

diagnoses = pd.DataFrame(index=pd.Index(sorted(dxf.hadm_id.unique()), name="hadm_id"))
diagnoses["primary_diagnosis"]   = principal.long_title
diagnoses["primary_icd_code"]    = principal.icd_code
diagnoses["primary_icd_version"] = principal.icd_version
diagnoses["secondary_diagnoses"] = secondary
diagnoses["secondary_diagnoses"] = diagnoses.secondary_diagnoses.apply(
    lambda v: v if isinstance(v, list) else [])
diagnoses["n_diagnoses_coded"]   = dxf.groupby("hadm_id").size()

# Some admissions carry secondary codes but no seq_num == 1 row. Saying so beats
# a blank cell, which reads as "not sick".
diagnoses["primary_diagnosis"] = diagnoses.primary_diagnosis.fillna(
    "No principal diagnosis coded for this admission")
diagnoses = diagnoses.reset_index()

report("admission diagnoses", diagnoses)
print(f"  admissions with a principal diagnosis: "
      f"{(~diagnoses.primary_diagnosis.str.startswith('No principal')).mean():.1%}")
print("\n  10 most common principal diagnoses:")
print(diagnoses.primary_diagnosis.value_counts().head(10).to_string())

save_cache(diagnoses, "phase1_diagnoses")
del dxf, dicd; gc.collect()


In [ ]:
# ---------------------------------------------------------------------------
# Export 2 - admission timeline, de-shifted to real calendar years
# ---------------------------------------------------------------------------
# MIMIC-IV pushes every patient's dates ~100 years into the future for
# de-identification, which is why raw admittimes read 2110-2201. The shift is a
# SINGLE PER-PATIENT offset, so intervals between a patient's own admissions are
# already exact - only the epoch is wrong.
#
# anchor_year_group is the true 3-year range that the shifted anchor_year maps
# to ("2008 - 2010" ... "2020 - 2022"). Subtracting (anchor_year - real anchor
# year) therefore returns every admission to the real collection window while
# preserving every interval to the day.
#
# HONESTY BOUND: this is a genuine de-shift, not an invention - but it resolves
# only to the 3-year group, so a recovered date is the right year +/- ~1.5, and
# the day-of-year is the shifted one. Treat it as a plausible real date, never
# as an exact calendar fact about a patient.
# ---------------------------------------------------------------------------

pt = pd.read_csv(tbl("patients"),
                 usecols=["subject_id", "anchor_year", "anchor_year_group"])

# "2008 - 2010" -> 2009. The midpoint spreads the residual error symmetrically
# instead of biasing every patient to the start of their group.
grp = pt.anchor_year_group.astype(str).str.extract(r"(\d{4})\D+(\d{4})").astype(float)
pt["real_anchor_year"] = ((grp[0] + grp[1]) / 2).round()
pt["shift_years"] = pt.anchor_year - pt.real_anchor_year

timeline = (X[["subject_id", "hadm_id", "admittime", "dischtime"]]
            .merge(pt, on="subject_id", how="left"))

# Every admission of a given patient gets the IDENTICAL offset, so applying the
# shift in whole days cannot disturb any within-patient interval.
shift_days = (timeline.shift_years * 365.2425).round()
unresolved = int(shift_days.isna().sum())
shift_days = shift_days.fillna(0)          # leave unresolvable rows untouched
                                           # rather than emitting NaT
timeline["admit_real"] = timeline.admittime - pd.to_timedelta(shift_days, unit="D")
timeline["disch_real"] = timeline.dischtime - pd.to_timedelta(shift_days, unit="D")
timeline["date_deshifted"] = shift_days.ne(0)

print(f"  shifted (raw MIMIC) : {timeline.admittime.min().date()} -> {timeline.dischtime.max().date()}")
print(f"  de-shifted          : {timeline.admit_real.min().date()} -> {timeline.disch_real.max().date()}")
print(f"  rows left un-shifted (no anchor_year_group): {unresolved:,}")

# Interval preservation is the property everything downstream depends on, so it
# is asserted here rather than assumed.
_chk = timeline.sort_values(["subject_id", "admittime"])
_a = _chk.groupby("subject_id").admittime.diff().dt.total_seconds()
_b = _chk.groupby("subject_id").admit_real.diff().dt.total_seconds()
_err = (_a - _b).abs().max()
print(f"  max within-patient interval drift: {0.0 if pd.isna(_err) else _err:.1f} seconds")
assert pd.isna(_err) or _err < 90000, "de-shift disturbed within-patient intervals"

# Self-check. MIMIC-IV was collected 2008-2022, so a correct de-shift puts
# essentially every admission inside that window. If this share is low the
# anchor_year_group parse failed and the dates must NOT be used - fall back to
# the raw shifted dates rather than shipping wrong ones.
WINDOW = (pd.Timestamp("2008-01-01"), pd.Timestamp("2022-12-31"))
in_window = timeline.admit_real.between(*WINDOW).mean()
print(f"  de-shifted dates inside the real 2008-2022 collection window: {in_window:.1%}")
if in_window < 0.90:
    print("  WARNING: de-shift looks wrong - inspect anchor_year_group before using these dates")

report("admission timeline", timeline)
save_cache(timeline, "phase1_timeline")
del pt; gc.collect()


## 7 &middot; Split **by patient**

A patient can contribute several admissions. Splitting rows at random puts the same person on
both sides and inflates every metric. `GroupShuffleSplit` on `subject_id` keeps each patient
wholly in one fold.

A three-way split is used because the calibrator must be fitted on data the classifier has
never seen &mdash; calibrating on the training fold reproduces the training-set optimism.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

y = X.readmit_30d.values
groups = X.subject_id.values
Xf = X[FEATURES]

gss = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
trainval_idx, test_idx = next(gss.split(Xf, y, groups))

gss2 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
tr_rel, cal_rel = next(gss2.split(Xf.iloc[trainval_idx], y[trainval_idx], groups[trainval_idx]))
train_idx = trainval_idx[tr_rel]
calib_idx = trainval_idx[cal_rel]

for name, idx in [("train", train_idx), ("calibration", calib_idx), ("test", test_idx)]:
    print(f"  {name:<12} rows={len(idx):>8,}  patients={pd.Series(groups[idx]).nunique():>7,}"
          f"  positives={y[idx].mean():.2%}")

overlap = set(groups[train_idx]) & set(groups[test_idx])
assert not overlap, f"patient leakage across folds: {len(overlap)}"
print("\n  no patient appears in more than one fold")

## 8 &middot; Train

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV

NUMERIC = [c for c in FEATURES if c not in CATEGORICALS]

# --- baseline: regularised logistic regression -------------------------------
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), NUMERIC),
    ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01), CATEGORICALS),
])
logit = Pipeline([("pre", pre),
                  ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", C=0.1))])

print("fitting logistic regression ...")
logit.fit(Xf.iloc[train_idx], y[train_idx])

# --- gradient boosting, native categorical support ---------------------------
cat_mask = [c in CATEGORICALS for c in FEATURES]
hgb = HistGradientBoostingClassifier(
    max_iter=400, learning_rate=0.06, max_leaf_nodes=31,
    min_samples_leaf=40, l2_regularization=1.0,
    categorical_features=cat_mask, early_stopping=True,
    validation_fraction=0.15, random_state=42,
)
print("fitting gradient boosting ...")
hgb.fit(Xf.iloc[train_idx], y[train_idx])

# --- calibrate on the held-out calibration fold ------------------------------
# Isotonic needs a few thousand rows; fall back to sigmoid on small cohorts.
method = "isotonic" if len(calib_idx) >= 2000 else "sigmoid"
print(f"calibrating with method={method} on {len(calib_idx):,} held-out rows")
hgb_cal = CalibratedClassifierCV(hgb, method=method, cv="prefit")
hgb_cal.fit(Xf.iloc[calib_idx], y[calib_idx])
print("done")

## 9 &middot; Evaluate

For a ~10% prevalence problem, **AUC-PR** and **calibration** matter more than AUC-ROC.
AUC-ROC stays flattering under imbalance; the precision-recall curve and the Brier score do not.

Published readmission models cluster around **AUC-ROC 0.71** (pooled, systematic review), so
anything far above that here should be treated as a leakage signal rather than a win.

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, brier_score_loss,
                             classification_report, confusion_matrix,
                             precision_recall_curve, roc_curve)
from sklearn.calibration import calibration_curve
import matplotlib.pyplot as plt

Xte, yte = Xf.iloc[test_idx], y[test_idx]
models = {
    "Logistic Regression": logit.predict_proba(Xte)[:,1],
    "HistGradientBoosting": hgb.predict_proba(Xte)[:,1],
    "HGB + calibration": hgb_cal.predict_proba(Xte)[:,1],
}

rows = []
for name, p in models.items():
    rows.append({
        "model": name,
        "AUC-ROC": roc_auc_score(yte, p),
        "AUC-PR":  average_precision_score(yte, p),
        "Brier":   brier_score_loss(yte, p),
        "mean_pred": p.mean(),
    })
res = pd.DataFrame(rows).set_index("model")
res["prevalence"] = yte.mean()
res["calib_ratio"] = res.mean_pred / res.prevalence
print(res.round(4).to_string())
print(f"\nbaseline AUC-PR (predicting prevalence) = {yte.mean():.4f}")
print("calib_ratio near 1.00 means predicted probabilities match reality.")

In [ ]:
best_name = "HGB + calibration"
p = models[best_name]

fig, ax = plt.subplots(1, 3, figsize=(16, 4.4))

fpr, tpr, _ = roc_curve(yte, p)
ax[0].plot(fpr, tpr, lw=2); ax[0].plot([0,1],[0,1],"--",c="grey",lw=1)
ax[0].set(title=f"ROC (AUC={roc_auc_score(yte,p):.3f})",
          xlabel="False positive rate", ylabel="True positive rate")

prec, rec, _ = precision_recall_curve(yte, p)
ax[1].plot(rec, prec, lw=2)
ax[1].axhline(yte.mean(), ls="--", c="grey", lw=1, label=f"baseline {yte.mean():.3f}")
ax[1].set(title=f"Precision-Recall (AP={average_precision_score(yte,p):.3f})",
          xlabel="Recall", ylabel="Precision"); ax[1].legend()

frac_pos, mean_pred = calibration_curve(yte, p, n_bins=10, strategy="quantile")
ax[2].plot(mean_pred, frac_pos, "o-", lw=2)
ax[2].plot([0,1],[0,1],"--",c="grey",lw=1)
ax[2].set(title="Calibration", xlabel="Predicted probability", ylabel="Observed frequency")
ax[2].set_xlim(0, max(mean_pred.max(), frac_pos.max())*1.1)
ax[2].set_ylim(0, max(mean_pred.max(), frac_pos.max())*1.1)

plt.tight_layout(); plt.show()

In [ ]:
# Operating point: choose the threshold by the recall the care team can staff for,
# not by argmax accuracy. A 10%-prevalence problem is worked as a triage queue.
TARGET_RECALL = 0.60
prec, rec, thr = precision_recall_curve(yte, p)
ok = np.where(rec[:-1] >= TARGET_RECALL)[0]
threshold = float(thr[ok[-1]]) if len(ok) else 0.5

pred = (p >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()

print(f"threshold for >={TARGET_RECALL:.0%} recall : {threshold:.4f}\n")
print(classification_report(yte, pred, target_names=["no readmit","readmit"], digits=3))
print(f"confusion matrix   TN={tn:,}  FP={fp:,}  FN={fn:,}  TP={tp:,}")
print(f"\nOperational read:")
print(f"  flagged per 1,000 discharges : {1000*(tp+fp)/len(yte):.0f}")
print(f"  of those, true readmissions  : {tp/(tp+fp):.1%}")
print(f"  readmissions missed          : {fn/(tp+fn):.1%}")

In [ ]:
from sklearn.inspection import permutation_importance

# Permutation importance on a subsample: model-agnostic and does not inherit
# the split-count bias of tree impurity importance.
sub = np.random.RandomState(0).choice(len(test_idx), size=min(4000, len(test_idx)), replace=False)
imp = permutation_importance(hgb, Xte.iloc[sub], yte[sub], n_repeats=5,
                             random_state=0, scoring="average_precision", n_jobs=-1)
top = (pd.DataFrame({"feature": FEATURES, "importance": imp.importances_mean})
         .sort_values("importance", ascending=False).head(20))
print(top.to_string(index=False))

plt.figure(figsize=(8, 6))
plt.barh(top.feature[::-1], top.importance[::-1])
plt.xlabel("drop in AUC-PR when shuffled"); plt.title("Top 20 features")
plt.tight_layout(); plt.show()

## 10 &middot; Export model card and calibrated model

In [ ]:
import joblib

pcal = models["HGB + calibration"]
card = {
    "architecture": "HistGradientBoostingClassifier + isotonic calibration",
    "dataset": "MIMIC-IV hosp",
    "trained_at": pd.Timestamp.utcnow().strftime("%Y-%m-%d"),
    "cohort": {
        "index_stays": int(len(X)),
        "patients": int(X.subject_id.nunique()),
        "exclusions": "in-hospital death, hospice discharge, observation stays",
        "label": "30-day unplanned readmission (elective returns excluded)",
    },
    "prevalence": round(float(yte.mean()), 4),
    "metrics": {
        "auc_roc":   round(float(roc_auc_score(yte, pcal)), 4),
        "auc_pr":    round(float(average_precision_score(yte, pcal)), 4),
        "brier":     round(float(brier_score_loss(yte, pcal)), 4),
        "threshold": round(threshold, 4),
        # precision/recall reported for the POSITIVE class at the chosen threshold
        "precision_readmit": round(float(tp/(tp+fp)) if (tp+fp) else 0.0, 4),
        "recall_readmit":    round(float(tp/(tp+fn)) if (tp+fn) else 0.0, 4),
    },
    "notes": [
        "Probabilities are calibrated; score may be read as a real probability.",
        "Split by patient (GroupShuffleSplit) - no subject appears in two folds.",
        "Single-centre data: readmissions to other hospitals are not observed, "
        "so the true rate is higher than measured.",
    ],
}
print(json.dumps(card, indent=2))

with open(os.path.join(CACHE_DIR, "model_card_phase1.json"), "w") as f:
    json.dump(card, f, indent=2)
joblib.dump({"model": hgb_cal, "features": FEATURES, "categoricals": CATEGORICALS,
             "threshold": threshold}, os.path.join(CACHE_DIR, "phase1_model.joblib"))
# ---------------------------------------------------------------------------
# Export 3 - feature spec for the serving API
# ---------------------------------------------------------------------------
# api/mimic_scoring.py reads this to build the manual-entry form: the exact
# feature order, the exact category levels seen in training, and the operating
# threshold. It was previously hand-maintained, which meant a retrain silently
# desynchronised the form from the model. Writing it here ties the two together.
#
# population_defaults are medians for callers that prefer imputation; the API
# default is honest missingness, since HistGradientBoosting handles NaN natively.
spec = {
    "model_version": "mimic-hgb-calibrated-v1",
    "features": list(FEATURES),
    "categoricals": {c: sorted(map(str, X[c].cat.categories)) for c in CATEGORICALS},
    "threshold": float(threshold),
    "population_defaults": {
        c: float(X[c].median())
        for c in FEATURES
        if c not in CATEGORICALS and pd.notna(X[c].median())
    },
    "prevalence": float(X.readmit_30d.mean()),
}
with open(os.path.join(CACHE_DIR, "mimic_feature_spec.json"), "w") as f:
    json.dump(spec, f, indent=2)
print(f"  mimic_feature_spec.json   {len(spec['features'])} features, "
      f"{len(spec['categoricals'])} categoricals")

print(f"\nwritten to {CACHE_DIR}/")
print("""
Download ALL of these from the Kaggle output panel:

  phase1_model.joblib        trained calibrated model
  model_card_phase1.json     metrics and cohort description
  mimic_feature_spec.json    feature order, category levels, threshold
  phase1_matrix.parquet      scored matrix  <-- NOW CARRIES hadm_id
  phase1_diagnoses.parquet   principal + secondary diagnoses per admission
  phase1_timeline.parquet    real (de-shifted) admit/discharge dates
  phase1_labs.parquet        lab aggregates   (unchanged)
  phase1_meds.parquet        med aggregates   (unchanged)
""")